# StreamForge — FLUX.1-dev in Pure Rust on Kaggle

Full BF16 FLUX.1-dev inference pipeline written in **pure Rust** using [candle](https://github.com/huggingface/candle).  
Streams the 24GB transformer **block-by-block** from disk through CPU RAM to GPU — peak VRAM ~4GB.

**Source code:** https://github.com/madtunebk/GPULoad

## What this notebook does
1. Mounts the pre-converted model dataset (flux_candle + vae_candle safetensors)
2. Downloads CLIP-L + T5-XXL weights from HuggingFace (~10GB, ~2 min)
3. Uses pre-built binaries OR builds from source if glibc mismatch
4. Runs the full pipeline: text_encoder → flux_gpu → vae_decode → PNG

## Required Kaggle Dataset
Add your dataset (containing `flux_candle.safetensors` and `vae_candle.safetensors`) via:  
**Add Data → Models → flux-dev-rust-candle** (by valicornea84)

It will be mounted at `/kaggle/input/models/valicornea84/flux-dev-rust-candle/other/default/1/`

## GPU
Enable GPU accelerator: **Settings → Accelerator → GPU T4 x2** (or P100)

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], 
                        capture_output=True, text=True)
print(result.stdout)

import os
print(f"HOME={os.environ.get('HOME')}")
print(f"CUDA available:", os.path.exists('/usr/local/cuda'))

## Step 1 — Check model dataset is mounted

## Step 1b — HuggingFace Authentication

FLUX.1-dev is a **gated model** — you must:
1. Accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev
2. Create a **classic** Read token at https://huggingface.co/settings/tokens  
   ⚠️ Fine-grained tokens block gated repos by default — use classic OR enable "Access to public gated repositories" in the fine-grained token settings
3. Add it as a Kaggle Secret: **Add-ons → Secrets → Add** with name `HF_TOKEN`

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # older HF versions
print("HF token loaded")

In [ ]:
import os

DATASET_DIR = "/kaggle/input/models/valicornea84/flux-dev-rust-candle/other/default/1"
FLUX_MODEL  = f"{DATASET_DIR}/flux_candle.safetensors"
VAE_MODEL   = f"{DATASET_DIR}/vae_candle.safetensors"

for path in [FLUX_MODEL, VAE_MODEL]:
    size = os.path.getsize(path) / 1e9
    print(f"OK  {path}  ({size:.1f} GB)")

## Step 2 — Download CLIP-L + T5-XXL from HuggingFace

These weights stay in the HF cache at `~/.cache/huggingface/` which matches the hardcoded paths in the Rust binaries.

> Takes ~2 minutes on Kaggle's network.

In [ ]:
!pip install -q huggingface_hub

# Download CLIP + T5 + tokenizers only (transformer comes from the Kaggle dataset)
from huggingface_hub import snapshot_download
import os

snapshot_download(
    repo_id="black-forest-labs/FLUX.1-dev",
    allow_patterns=["text_encoder/**", "text_encoder_2/**", "tokenizer/**", "tokenizer_2/**"],
    token=os.environ["HF_TOKEN"],
    ignore_patterns=["*.bin"],   # skip pytorch_model.bin if present, keep safetensors only
)

print("Done.")

## Step 3 — Set up workspace and symlink models

In [ ]:
import os, subprocess, shutil

WORKDIR = "/kaggle/working/streamforge"
os.makedirs(f"{WORKDIR}/models/tokenizer", exist_ok=True)
os.makedirs(f"{WORKDIR}/temp",             exist_ok=True)
os.makedirs(f"{WORKDIR}/loras",            exist_ok=True)
os.makedirs(f"{WORKDIR}/bin",              exist_ok=True)

# Symlink pre-converted models
for fname, src in [("flux_candle.safetensors", FLUX_MODEL), ("vae_candle.safetensors", VAE_MODEL)]:
    dst = f"{WORKDIR}/models/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"linked  {dst}")

# Clone repo once — provides tokenizer + pre-built binaries
repo_dir = "/kaggle/working/GPULOAD"
if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/madtunebk/GPULoad.git", repo_dir], check=True)
print(f"repo    {repo_dir}")

# Copy CLIP tokenizer from repo
tok_dst = f"{WORKDIR}/models/tokenizer/tokenizer.json"
if not os.path.exists(tok_dst):
    shutil.copy2(f"{repo_dir}/models/tokenizer/tokenizer.json", tok_dst)
print(f"copied  {tok_dst}")

# Copy + chmod pre-built binaries
BIN_DST = f"{WORKDIR}/bin"
for b in ["text_encoder", "flux_gpu", "vae_decode"]:
    dst = f"{BIN_DST}/{b}"
    shutil.copy2(f"{repo_dir}/dist/{b}", dst)
    os.chmod(dst, 0o755)
print(f"binaries in {BIN_DST}")

# Smoke test
r = subprocess.run([f"{BIN_DST}/flux_gpu", "--help"], capture_output=True, text=True)
print(r.stdout.splitlines()[0] if r.stdout else r.stderr.splitlines()[0])


## Step 4 — Build from source on Kaggle (native, CUDA_COMPUTE_CAP=70)

Installs Rust toolchain and compiles the 3 binaries directly on Kaggle.  
Takes ~5–8 min. Skip if pre-built binaries from Step 3 already work.


In [ ]:
import os, shutil, glob, time

repo_dir = "/kaggle/working/GPULOAD"
BIN_DST  = f"{WORKDIR}/bin"

# --- Install Rust ---
!curl https://sh.rustup.rs -sSf | sh -s -- -y --default-toolchain stable --profile minimal
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

# --- Find libcuda.so and symlink if needed ---
found_cuda_dir = "/usr/local/cuda/lib64/stubs"
for d in ["/usr/local/cuda/compat", "/usr/lib/x86_64-linux-gnu", "/usr/lib/x86_64-linux-gnu/nvidia"]:
    hits = glob.glob(f"{d}/libcuda.so*")
    if hits:
        found_cuda_dir = d
        stub = f"{d}/libcuda.so"
        if not os.path.exists(stub):
            try:
                os.symlink(hits[0], stub)
            except PermissionError:
                os.makedirs("/tmp/cuda_stubs", exist_ok=True)
                tgt = "/tmp/cuda_stubs/libcuda.so"
                if not os.path.exists(tgt): os.symlink(hits[0], tgt)
                found_cuda_dir = "/tmp/cuda_stubs"
        print(f"libcuda dir: {found_cuda_dir}")
        break

# --- Set build env vars ---
os.environ["CUDA_COMPUTE_CAP"] = "70"
os.environ["CUDA_PATH"]        = "/usr/local/cuda"
os.environ["CUDA_ROOT"]        = "/usr/local/cuda"
os.environ["RUSTFLAGS"]        = (
    f"-C link-arg=-L{found_cuda_dir} "
    f"-C link-arg=-L/usr/local/cuda/lib64/stubs "
    f"-C link-arg=-L/usr/local/cuda/lib64"
)

# --- Build ---
t0 = time.time()
!cd /kaggle/working/GPULOAD && ~/.cargo/bin/cargo build --release --locked --features cuda
print(f"Build done in {time.time()-t0:.0f}s")

# --- Copy binaries ---
for b in ["text_encoder", "flux_gpu", "vae_decode"]:
    shutil.copy2(f"{repo_dir}/target/release/{b}", f"{BIN_DST}/{b}")
    os.chmod(f"{BIN_DST}/{b}", 0o755)
    print(f"copied  {BIN_DST}/{b}")

!{BIN_DST}/flux_gpu --help | head -1


## Step 5 — Run the pipeline

Edit `PROMPT`, `WIDTH`, `HEIGHT`, `STEPS` below.

In [ ]:

import os

checks = [
    f"{WORKDIR}/models/flux_candle.safetensors",
    f"{WORKDIR}/models/vae_candle.safetensors",
    f"{WORKDIR}/models/tokenizer/tokenizer.json",
    f"{WORKDIR}/temp",
    f"{WORKDIR}/bin/text_encoder",
    f"{WORKDIR}/bin/flux_gpu",
    f"{WORKDIR}/bin/vae_decode",
]

all_ok = True
for p in checks:
    exists = os.path.exists(p)
    status = "OK " if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"[{status}] {p}")

if not all_ok:
    raise RuntimeError("Pre-flight check failed — fix missing paths above before running pipeline")
else:
    print("\nAll checks passed.")


In [ ]:
PROMPT   = "anime2, masterpiece, best quality, 1girl, white hair, red eyes, holding a glowing blue sword, fantasy armor, epic background"
WIDTH    = 768
HEIGHT   = 1024
STEPS    = 20
GUIDANCE = 3.5
SEED     = 42
LORA     = None   # e.g. "/kaggle/input/models/valicornea84/flux-dev-rust-candle/other/default/1/loras/my.safetensors"
LORA_SCALE = 1.0

In [ ]:
import subprocess, os, time

env = os.environ.copy()
env["PATH"] = f"{BIN_DST}:{env['PATH']}"

def run(cmd, **kwargs):
    print(f"$ {' '.join(cmd)}")
    t0 = time.time()
    result = subprocess.run(cmd, cwd=WORKDIR, env=env,
                            capture_output=True, text=True, **kwargs)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    print(f"  → {time.time()-t0:.1f}s  (exit {result.returncode})")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")

# --- 1. text_encoder (CPU — T5 is always CPU anyway, CLIP is only 246MB) ---
te_cmd = [f"{BIN_DST}/text_encoder", PROMPT, "--device", "cpu"]
if LORA:
    te_cmd += ["--lora", LORA, "--lora-scale", str(LORA_SCALE)]
run(te_cmd)

# --- 2. flux_gpu ---
fg_cmd = [f"{BIN_DST}/flux_gpu",
          "--width",    str(WIDTH),
          "--height",   str(HEIGHT),
          "--steps",    str(STEPS),
          "--guidance", str(GUIDANCE),
          "--seed",     str(SEED)]
if LORA:
    fg_cmd += ["--lora", LORA, "--lora-scale", str(LORA_SCALE)]
run(fg_cmd)

# --- 3. vae_decode ---
run([f"{BIN_DST}/vae_decode"])


## Result

In [ ]:
from IPython.display import Image, display
import shutil, os

output_src = f"{WORKDIR}/temp/output_rust.png"
output_dst = "/kaggle/working/output_rust.png"
shutil.copy2(output_src, output_dst)

display(Image(filename=output_dst))
print(f"Saved to {output_dst}")